# Predictive Modeling Using Machine Learning
## Titanic Passenger Survival Prediction

**Submitted by:** ____________________  
**Course:** ____________________  
**Date:** ____________________

### Objective
Build and evaluate a machine-learning model that predicts whether a Titanic passenger survived, using passenger class, gender, age, family information, fare, and port of embarkation.


## 1. Import Libraries
Run this once before starting if needed:

`pip install pandas numpy matplotlib seaborn scikit-learn jupyter`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

sns.set_theme(style='whitegrid', palette='deep')


## 2. Load the Dataset
The supplied file is a small Titanic-style educational dataset. It intentionally contains missing values, duplicate rows, and a few fare outliers so the cleaning process can be demonstrated.


In [ ]:
data = pd.read_csv('dataset/raw_titanic_sample.csv')
data.head()


## 3. Initial Data Exploration


In [ ]:
print('Dataset shape:', data.shape)
print('\nColumn data types:')
print(data.dtypes)
print('\nMissing values:')
print(data.isnull().sum())
print('\nDuplicate rows:', data.duplicated().sum())
data.describe(include='all')


## 4. Clean the Data
- Remove duplicate passengers
- Fill numerical missing values with the median
- Fill missing embarkation values with the mode
- Create `FamilySize` and `IsAlone` features


In [ ]:
clean = data.drop_duplicates(subset='PassengerId').copy()
clean['Age'] = clean['Age'].fillna(clean['Age'].median())
clean['Fare'] = clean['Fare'].fillna(clean['Fare'].median())
clean['Embarked'] = clean['Embarked'].fillna(clean['Embarked'].mode()[0])
clean['FamilySize'] = clean['SibSp'] + clean['Parch'] + 1
clean['IsAlone'] = (clean['FamilySize'] == 1).astype(int)

print('Clean dataset shape:', clean.shape)
print('Remaining missing values:', clean.isnull().sum().sum())
clean.to_csv('dataset/cleaned_titanic_data.csv', index=False)


## 5. Detect and Treat Outliers
The IQR method identifies unusually high or low fares. Instead of deleting records, fare values are capped at the upper and lower IQR limits. This retains passengers while reducing the effect of extreme fares.


In [ ]:
q1, q3 = clean['Fare'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
outliers = clean[(clean['Fare'] < lower_limit) | (clean['Fare'] > upper_limit)]
print('Fare outliers found:', len(outliers))
clean['Fare'] = clean['Fare'].clip(lower=lower_limit, upper=upper_limit)

plt.figure(figsize=(8, 4))
sns.boxplot(x=clean['Fare'], color='#60a5fa')
plt.title('Fare Distribution After Outlier Treatment')
plt.show()


## 6. Exploratory Data Analysis and Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=clean, x='Sex', hue='Survived', ax=axes[0])
axes[0].set_title('Survival Count by Gender')
sns.countplot(data=clean, x='Pclass', hue='Survived', ax=axes[1])
axes[1].set_title('Survival Count by Passenger Class')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(data=clean, x='Age', hue='Survived', bins=22, kde=True, element='step')
plt.title('Age Distribution by Survival')
plt.show()


## 7. Prepare Data for Machine Learning
`Survived` is the target. The remaining selected columns are the features used to predict it.


In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']
X = clean[features]
y = clean['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Embarked']

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])


## 8. Train the Model
A Random Forest classifier is used because it handles non-linear patterns and mixed passenger features well.


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, predictions)
print(f'Test accuracy: {accuracy:.2%}')
print('\nClassification report:')
print(classification_report(y_test, predictions, target_names=['Not Survived', 'Survived']))


## 9. Evaluate Model Performance
The confusion matrix shows correct and incorrect predictions. The ROC curve shows how well the model separates the two outcome classes across probability thresholds.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, predictions), display_labels=['Not Survived', 'Survived']).plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_test, probabilities)
auc = roc_auc_score(y_test, probabilities)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc:.2f})', linewidth=2)
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## 10. Feature Importance


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importance = model.named_steps['classifier'].feature_importances_
importance_table = pd.DataFrame({'Feature': feature_names, 'Importance': importance}).sort_values('Importance', ascending=False)
print(importance_table.head(10))

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_table.head(10), x='Importance', y='Feature', color='#2563eb')
plt.title('Top 10 Feature Importances')
plt.show()


## 11. Insights and Conclusion

### Key Insights
1. Gender is strongly related to survival: female passengers have a higher survival rate in this dataset.
2. First-class passengers generally survive at a higher rate than third-class passengers.
3. Age, fare, and family size provide additional useful information for prediction.
4. The final Random Forest model is evaluated with accuracy, a confusion matrix, and an ROC curve rather than accuracy alone.

### Conclusion
This project cleaned a raw passenger dataset, explored important patterns, created meaningful visualizations, and trained a supervised machine-learning model. The model can estimate the survival outcome of new passengers using the selected features. Results should be interpreted as patterns in this educational sample, not as a complete historical analysis of all Titanic passengers.
